## 1. Préparation de l'environnement et des données ESS

Je reprends la base d'avant en chargeant nos fichiers


In [37]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import random

In [64]:

# Chargement de la liste ESS (on ne garde que le SIREN pour la jointure)
df_ess_ref = pd.read_csv('..\\..\\data\\raw\\entreprisesess.csv', usecols=['SIREN'], dtype={'SIREN': str}, nrows=10000)
ess_sirens = set(df_ess_ref['SIREN'])

print(f"Nombre de structures ESS référencées : {len(ess_sirens)}")

Nombre de structures ESS référencées : 10000


In [65]:
file = pq.ParquetFile('..\\..\\data\\raw\\StockUniteLegale_utf8.parquet')


## 2. Lecture optimisée du stock INSEE (Parquet) pour 10 row

Pour ne pas charger 30 Millions de lignes, nous allons utiliser une lecture sélective. Choix 2 :  Prendre les catégorie juridique [9220, 6316, 9300] qui sont associations, coopératives ou fondations, en plsu de travailler sur 10 chunks. Nous avons donc déjà un laissé pour compte : les ESUS

In [40]:
# Liste des colonnes nécessaires pour la cartographie
cols_to_keep = [
    'siren', 'denominationUniteLegale', 'nomUniteLegale', 
    'activitePrincipaleUniteLegale', 'trancheEffectifsUniteLegale', 'categorieJuridiqueUniteLegale'
]


In [41]:
ESS_CODES = [
    9220,  # Associations
    6316, 6317, 6318,  # Coopératives
    9300   # Fondations
]


In [42]:
codes_naf_numerique = [
    '62.01Z', '58.29A', '58.29B', '58.29C',
    '63.11Z'
]

In [53]:
ess_numerique_chunks = []

n_groups = min(10, file.num_row_groups)

for i in range(n_groups):
    print(f"Lecture row group {i+1}/{n_groups}")
    
    table = file.read_row_group(i, columns=cols_to_keep)
    df_chunk = table.to_pandas()
    
    df_filtered = df_chunk[
        (df_chunk['categorieJuridiqueUniteLegale'].isin(ESS_CODES)) &
        (df_chunk['activitePrincipaleUniteLegale'].isin(codes_naf_numerique))
    ]
    
    if not df_filtered.empty:
        ess_numerique_chunks.append(df_filtered)

df_ess_numerique_partiel = pd.concat(ess_numerique_chunks, ignore_index=True)

print(f"Nombre total (sur 10 row groups) : {len(df_ess_numerique_partiel)}")


Lecture row group 1/10
Lecture row group 2/10
Lecture row group 3/10
Lecture row group 4/10
Lecture row group 5/10
Lecture row group 6/10
Lecture row group 7/10
Lecture row group 8/10
Lecture row group 9/10
Lecture row group 10/10
Nombre total (sur 10 row groups) : 22


## 3. lecture et filtres pour tout 

ici je veux juste voir le nombre de structures que je peux récupérer avec ces 3 filtres : 
1) Codes ESS donc la structure juridique
2) 

In [ ]:

ess_numerique_chunks = []

# On utilise le nombre total de groupes de lignes du fichier
total_groups = file.num_row_groups

for i in range(total_groups):
    if (i + 1) % 50 == 0:  # Optionnel : affiche un log toutes les 50 itérations pour ne pas saturer la console
        print(f"Progression : {i+1}/{total_groups} row groups traités")
    
    # Lecture du groupe i
    table = file.read_row_group(i, columns=cols_to_keep)
    df_chunk = table.to_pandas()
    
    # Filtrage restrictif : ESS + Codes NAF Numériques
    df_filtered = df_chunk[
        (df_chunk['categorieJuridiqueUniteLegale'].isin(ESS_CODES)) &
        (df_chunk['activitePrincipaleUniteLegale'].isin(codes_naf_numerique))
    ].copy()
    
    if not df_filtered.empty:
        # Nettoyage immédiat pour la lisibilité (remplacement des sauts de ligne par /)
        if 'TYPE_ACT' in df_filtered.columns:
            df_filtered['TYPE_ACT'] = df_filtered['TYPE_ACT'].str.replace(r';\n', ' / ', regex=True)
        
        ess_numerique_chunks.append(df_filtered)

# Concaténation de tous les morceaux
df_ess_numerique_complet = pd.concat(ess_numerique_chunks, ignore_index=True)

# Suppression des lignes vides résiduelles (comme l'index 1633 vu précédemment)
df_ess_numerique_complet = df_ess_numerique_complet.dropna(subset=['activitePrincipaleUniteLegale'])

print(f"Traitement terminé. Nombre total d'entités trouvées : {len(df_ess_numerique_complet)}")

In [52]:
df_ess_numerique_complet.to_csv('C:\\Users\\Utilisateur\\OneDrive\\Documents\\centrale Lille + sc po\\Projet SOGA\\ess-numerique-database\\data\\processed\\ess_numerique_complet1.csv', index=False)    